# LANGFENS — RQ1: P-SUP B-Control Preflight Verification on Kaggle Tesla T4x2

Private execution notebook under CC BY-NC 4.0.
Audits hardware, secrets, Git HEAD provenance, Fold 1 staging and speaker isolation, and private HF remote mirror readiness.
STRICTLY FAILS CLOSED: zero training, optimizer steps, or model checkpoint generation.


In [ ]:
# Phase 1: Hardware & Environment Verification
import subprocess, sys

subprocess.run(["nvidia-smi"], check=True)

import torch
print(f"[GPU] CUDA Available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("FAIL-CLOSED: CUDA GPU is not available on this runtime!")
print(f"[GPU] Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"[GPU {i}]: {torch.cuda.get_device_name(i)}")
assert "Tesla T4" in torch.cuda.get_device_name(0), (
    f"FAIL-CLOSED: Expected Tesla T4 accelerator, got {torch.cuda.get_device_name(0)}"
)

import yaml, transformers, huggingface_hub
print(f"[ENV] PyTorch version: {torch.__version__}")
print(f"[ENV] Transformers version: {transformers.__version__}")
print(f"[ENV] Hugging Face Hub version: {huggingface_hub.__version__}")


In [ ]:
# Phase 2: Credentials & Secrets Resolution (Strict Kaggle Secrets Path)
import os

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token and token.strip():
        os.environ["HF_TOKEN"] = token.strip()
        print("[AUTH] Successfully loaded HF_TOKEN from Kaggle Secrets.")
except Exception as e:
    print(f"[AUTH NOTE] Kaggle Secrets UserSecretsClient query: {e}")

if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    raise RuntimeError(
        "FAIL-CLOSED: HF_TOKEN is required in Kaggle Secrets (or environment) for private remote mirroring. "
        "Attach HF_TOKEN under Add-ons -> Secrets in the Kaggle notebook."
    )
print("[AUTH] Verified HF_TOKEN presence in environment for private remote mirroring.")


In [ ]:
# Phase 3: Clone, Checkout Reviewed Training Commit & Verify Tree
import os, subprocess, sys
from pathlib import Path

repo_dir = Path("/kaggle/working/repo")
if not repo_dir.exists():
    print("[GIT] Cloning repository into /kaggle/working/repo...")
    subprocess.run(
        ["git", "clone", "https://github.com/minkhoaa/Langfens-RQ1-Speech-Research.git", str(repo_dir)],
        check=True
    )

os.chdir(str(repo_dir))
subprocess.run(["git", "fetch", "--all"], cwd=repo_dir, check=True)

PINNED_COMMIT_SHA = "56bf79535e4f697b2eea05594d5d81153e04dbff"
print(f"[GIT] Checking out pinned commit: {PINNED_COMMIT_SHA}...")
subprocess.run(["git", "checkout", "--detach", PINNED_COMMIT_SHA], cwd=repo_dir, check=True)

actual_head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo_dir, text=True).strip().lower()
print(f"[GIT] Verified Commit HEAD: {actual_head}")
assert actual_head == PINNED_COMMIT_SHA.lower(), (
    f"FAIL-CLOSED: Commit SHA mismatch: actual {actual_head} != expected {PINNED_COMMIT_SHA}"
)
print("[GIT] Repository verified clean at pinned commit.")

sys.path.insert(0, str(repo_dir / "src"))
sys.path.insert(0, str(repo_dir))


In [ ]:
# Phase 4: Staging Audit & Test Speaker Isolation Verification
import hashlib, zipfile
from pathlib import Path

staging_dir = Path("/kaggle/input/langfens-rq1-fold1-staging")
assert staging_dir.is_dir(), f"FAIL-CLOSED: Staging directory not found: {staging_dir}"

expected_sha = "b750eb3bf3971afccd9ab9058a64fa6b3f5781399b9ef0f1294d29bf580784a9"
candidate_files = [
    staging_dir / "authorized_RQ1-TRN-02A_fold1_train_val_only.bin",
    staging_dir / "authorized_RQ1-TRN-02A_fold1_train_val_only.zip"
]
archive_path = next((p for p in candidate_files if p.is_file()), None)
assert archive_path is not None, f"FAIL-CLOSED: Fold 1 archive not found in {staging_dir}"

h = hashlib.sha256()
with open(archive_path, "rb") as f:
    while chunk := f.read(1048576):
        h.update(chunk)
actual_sha = h.hexdigest()
print(f"[STAGING] Archive SHA-256: {actual_sha}")
assert actual_sha == expected_sha, f"FAIL-CLOSED: Archive SHA mismatch: {actual_sha} != {expected_sha}"

sealed_test_speakers = {"ABA", "ASI", "EBVS", "HJK", "LXC", "PNV"}
with zipfile.ZipFile(archive_path, "r") as zf:
    namelist = zf.namelist()
for name in namelist:
    for tspk in sealed_test_speakers:
        assert tspk not in name, f"FAIL-CLOSED: Test speaker leak detected: {name}"
print("[STAGING] Verified 100% test speaker isolation (0 test speakers in archive).")


In [ ]:
# Phase 5: Remote Persistence Verification Probe
from audit_scripts.kaggle_checkpoint_sync import KaggleRemoteMirror

remote_mirror = KaggleRemoteMirror(
    backend="huggingface",
    remote_target="minkhoaa/langfens-rq1-cross-fold-checkpoints",
    auth_token=os.environ["HF_TOKEN"],
    require_verification=True,
    strict_secrets_only=True
)
ok, msg = remote_mirror.verify_readiness()
assert ok, f"FAIL-CLOSED: Remote mirror probe failed: {msg}"
print(f"[REMOTE MIRROR] Remote persistence verified: {msg}")

print("=" * 80)
print("[PREFLIGHT COMPLETE] Preflight verification PASSED for P-SUP B-Control.")
print("[PREFLIGHT COMPLETE] Zero training, optimizer steps, or model checkpoint updates executed.")
print("=" * 80)
